<a href="https://colab.research.google.com/github/TonmoyTalukder/Bangla-Key2Text/blob/main/Bangla-KeywordExtractor/Keyword_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Read Dataset and Installs

In [ ]:
!pip install transformers

In [ ]:
!pip install git+https://github.com/csebuetnlp/normalizer

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Cloning https://github.com/csebuetnlp/normalizer to /tmp/pip-req-build-j94mgxex
  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/normalizer /tmp/pip-req-build-j94mgxex
  Resolved https://github.com/csebuetnlp/normalizer to commit d80c3c484e1b80268f2b2dfaf7557fe65e34f321
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for normalizer: filename=normalizer-0.0.1-py3-none-any.whl size=6860 sha256=baa842eb28f74a6e297e67dee2e895f4f1af1c9f06537b3617129f1599c62208
  Stored in directory: /tmp/pip-ephem-wheel-cache-2wyhgmw7/wheels/2e/79/9c/cd96d490298305d51d2da11484bb2c25fd1f759a6906708282
  Created wheel for emoji

In [ ]:
import pandas as pd

df0 = pd.read_csv("/content/drive/MyDrive/Shibli Sir/Treasure From Sir/8MBanglaSentences.csv")

In [ ]:
df0

,Sentence
0,"এ কারণে এসএ গেমস, এশিয়ান কিংবা অলিম্পিক গেমসের..."
1,"পর্যায়ক্রমে পাটুরিয়া-দৌলতদিয়া, আরিচা-নগরবাড়ী, ..."
2,গাজর-টমেটো বিক্রির টাকায় কেনেন ৫৪ শতক জমি।
3,"এর মধ্যে নবম ওয়েজ বোর্ড ঘোষণা কতটুকু যৌক্তিক,..."
4,কী এক ভিন্ন মাদকতা ও লাবণ্যতা তার সমস্ত কিছুতে!
...,...
8060027,সব ঠিক হয়ে যাবে।’দুটি বছর ভালোই ছিলেন মাশরাফি।
8060028,প্রথম ভারতীয় নারী হিসেবে কুস্তিতে পদক জিতলেন ত...
8060029,এ ছাড়া তাঁর পরিবারের আরও চার সদস্য মুক্তিযুদ্ধ...
8060030,গাজীপুরের কালিয়াকৈর উপজেলা বিএনপির কমিটি গতকা...


In [ ]:
df0['Sentence'][2341]

'বাবুরা অফিসে ঢোকার আগেই বগলে ঝাড়ু, কোমরে ঝাড়ন গুঁজে সেখানে আসে মেয়েটি।'

In [ ]:
# df = df0.head(500000)
# df  = df0.loc[500000:1600000]
df  = df0.head(3200000)
df

In [ ]:
df['Sentence'][1000001]

'সেই সঙ্গে চলেছে গানবাজনার চর্চা।'

## Generate Keywords and add to dataset

### Initials

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load pre-trained model and tokenizer
model_name = 'csebuetnlp/banglabert'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def score_words(sentence):
    # Tokenize input sentence
    # input_ids = torch.tensor([tokenizer.encode(sentence, max_length=512, padding='max_length', truncation=True)])
    input_ids = torch.tensor([tokenizer.encode(sentence)])

    # Generate word embeddings
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        embeddings = outputs.last_hidden_state.squeeze(0)

    # Calculate cosine similarity between each word and the mean of all word embeddings
    mean_embedding = embeddings.mean(dim=0)
    word_scores = []
    for i in range(embeddings.size(0)):
        cos_sim = torch.nn.functional.cosine_similarity(embeddings[i], mean_embedding, dim=0)
        word_scores.append((tokenizer.decode(input_ids[0][i]), cos_sim.item()))

    return word_scores

# Clean Sentence
def clean(sen):
    strs = ''
    for i in range(len(sen)):
        if sen[i] != '।':
            strs = strs + sen[i]
    return strs.split()

Some weights of the model checkpoint at csebuetnlp/banglabert were not used when initializing ElectraModel: ['discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense.bias']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
import random

def print_top_values(data):
    data_shuffled = random.sample(data, len(data))  # shuffle the list
    top_values = int(len(data_shuffled) * 0.6)  # calculate the number of top values to print
    if top_values < 10:
        top_values = int(len(data_shuffled) * 0.7)
    if top_values < 4:
        top_values = int(len(data_shuffled) * 0.8)
    finalLst = []
    for i in range(top_values):
        finalLst.append(data_shuffled[i][0])

    return finalLst

def keysOfSentence(sentence):

    lst = clean(sentence)
    lst2 = []
    final = []
    word_scores = score_words(sentence)

    # Get the Final Words with score
    for i in range(len(word_scores)):
        if i > 0:
            if word_scores[i][0] != '।':
                lst2.append(word_scores[i][1])
    for i in range(len(lst)):
        final.append(tuple([lst[i], lst2[i]]))

    # Thresholding of words
    finalKeysLst = print_top_values(final)

    return finalKeysLst

In [ ]:
text = df['Sentence'][500000]
keysOfSentence(text)

['সাগরে', 'রাখা', 'জাহাজে', 'ছিল']

In [ ]:
df['Sentence'][500004]

'আবুল কালাম আজাদ বলেন, গত শনিবার থেকে ৫-১২ বছর বয়সী সব শিশুকে কৃমিনাশক ট্যাবলেট খাওয়ানো শুরু হয়েছে।'

### 1700K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 1599999 and i < 1700000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        finally:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF1700K.csv", index=False)

### 1800K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 1699999 and i < 1800000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        finally:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF1800K.csv", index=False)

### 1900K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 1799999 and i < 1900000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        finally:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF1900K.csv", index=False)

### 2000K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    # if i > 899999 and i < 950000:
    if i > 1899999 and i < 2000000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        finally:
            continue

# genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF950K.csv", index=False)
genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2000K.csv", index=False)

### 2100K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 1999999 and i < 2100000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        finally:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2100K.csv", index=False)

### 2200K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2099999 and i < 2200000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2200K.csv", index=False)

### 2300K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2199999 and i < 2300000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2300K.csv", index=False)

### 2400K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2299999 and i < 2400000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2400K.csv", index=False)

### 2500K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2399999 and i < 2500000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2500K.csv", index=False)

### 2600K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2499999 and i < 2600000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2600K.csv", index=False)

### 2700K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2599999 and i < 2700000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2700K.csv", index=False)

### 2800K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2699999 and i < 2800000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2800K.csv", index=False)

### 2900K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2799999 and i < 2900000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF2900K.csv", index=False)

### 3000K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2899999 and i < 3000000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF3000K.csv", index=False)

### 3100K

In [ ]:
genDf = pd.DataFrame(columns = ["bn_Keywords", "bn_Sentence"])
rows = df.shape[0]

for i in range(rows):
    if i > 2999999 and i < 3100000:

        bn_text = df['Sentence'][i]

        try:
            bn_keywords = keysOfSentence(bn_text)
            bn_strlist = bn_keywords
            genDf.loc[i] = [bn_strlist, df['Sentence'][i]]
            print(i, ' ', bn_strlist)
        except:
            continue

genDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF3100K.csv", index=False)

Streaming output truncated to the last 5000 lines.
3094999   ['হয়েছে', 'অগ্রণী', 'নির্ধারণ', 'এবারের', 'বিষয়বস্তু', 'করা', '‘নতুন']
3095000   ['পারে', 'তা', 'আজহার', 'হতে']
3095001   ['তাঁর', 'নিয়ে', 'এখন', 'দেখার', 'আছেন', 'বর্তমানে', 'লেখান', 'নাডোয়াইন', 'কি', 'জনসন', 'ছবি', 'রাজনীতিতে']
3095002   ['খই', 'দুঃখ-দুর্দশার', 'বসে', 'জনগণের', 'আগামী', 'হয়ে', 'ফুটিয়ে', 'জনপ্রতিনিধিরা', 'কথা’সিএনজি', 'প্রতিশ্রুতির', 'ভুলে', 'চৌধুরী']
3095003   ['হাসি', 'বড়', 'এই', 'গৌরবের,', 'এক', 'সাফল্য', 'হাসি']
3095004   ['উপজেলার', 'সড়ক', 'গোবিন্দগঞ্জ', 'আরোহী', 'নছিমনের', 'শনিবার', 'গত', 'কালীতলা', 'গাইবান্ধার', 'নিহত']
3095005   ['ওই', 'স্ত্রী', 'রহমান', 'রাজিয়া', 'অজ্ঞাতদের', 'বিরুদ্ধে', 'ঘটনায়']
3095006   ['ইট', 'করছিলেন', 'স্থানীয়', 'দিকে', 'সড়ক', 'বের', 'লোকজন', 'চেষ্টা']
3095007   ['আবার', 'অতি', 'অহেতুক', 'নয়']
3095008   ['তৃতীয়', 'ভূইয়াদৈনিক', 'হাবীবুর', 'হাবীবুর', 'জ্যেষ্ঠ', 'ইনকিলাব-এর', 'সহসম্পাদক', 'রহমান', 'সাবেক', 'আজ']
3095009   []
3095010   ['সম্ভব', 'নেওয়া', 'তবে', 'প্রতিরোধ',

### Merge Dataset

In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genDF'

dff0 = pd.read_csv(path + '50K.csv')
dff1 = pd.read_csv(path + '100K.csv')
dff2 = pd.read_csv(path + '150K.csv')
dff3 = pd.read_csv(path + '200K.csv')
dff4 = pd.read_csv(path + '250K.csv')
dff5 = pd.read_csv(path + '300K.csv')
dff6 = pd.read_csv(path + '350K.csv')
dff7 = pd.read_csv(path + '400K.csv')
dff8 = pd.read_csv(path + '450K.csv')
dff9 = pd.read_csv(path + '500K.csv')
dff10 = pd.read_csv(path + '550K.csv')
dff11 = pd.read_csv(path + '600K.csv')
dff12 = pd.read_csv(path + '650K.csv')
dff13 = pd.read_csv(path + '700K.csv')
dff14 = pd.read_csv(path + '750K.csv')
dff15 = pd.read_csv(path + '800K.csv')
dff16 = pd.read_csv(path + '850K.csv')
dff17 = pd.read_csv(path + '900K.csv')
dff18 = pd.read_csv(path + '950K.csv')
dff19 = pd.read_csv(path + '1000K.csv')
dff20 = pd.read_csv(path + '1100K.csv')
dff21 = pd.read_csv(path + '1200K.csv')
dff22 = pd.read_csv(path + '1300K.csv')
dff23 = pd.read_csv(path + '1400K.csv')
dff24 = pd.read_csv(path + '1500K.csv')
dff25 = pd.read_csv(path + '1600K.csv')
dff26 = pd.read_csv(path + '1700K.csv')
dff27 = pd.read_csv(path + '1800K.csv')
dff28 = pd.read_csv(path + '1900K.csv')
dff29 = pd.read_csv(path + '2000K.csv')
dff30 = pd.read_csv(path + '2100K.csv')
dff31 = pd.read_csv(path + '2200K.csv')
dff32 = pd.read_csv(path + '2300K.csv')
dff33 = pd.read_csv(path + '2400K.csv')
dff34 = pd.read_csv(path + '2500K.csv')
dff35 = pd.read_csv(path + '2600K.csv')
dff36 = pd.read_csv(path + '2700K.csv')
dff37 = pd.read_csv(path + '2800K.csv')
dff38 = pd.read_csv(path + '2900K.csv')
dff39 = pd.read_csv(path + '3000K.csv')
dff40 = pd.read_csv(path + '3100K.csv')

frames = [dff0, dff1, dff2, dff3, dff4, dff5, dff6, dff7, dff8, dff9, dff10, dff11, dff12, dff13, dff14, dff15, dff16, dff17, dff18, dff19, dff20, dff21, dff22, dff23, dff24, dff25, dff26, dff27, dff28, dff29, dff30, dff31, dff32, dff33, dff34, dff35, dff36, dff37, dff38, dff39, dff40]
newDf = pd.concat(frames)
mergedDf = newDf.reset_index(drop=True)
mergedDf.to_csv("/content/drive/MyDrive/Shibli Sir/Multi-Modal Research Team/Work 0: Keywords to Text in Bengali/DataV3/genMergedDfV3000Plus.csv", index=False)

In [ ]:
mergedDf

In [ ]:
mergedDf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3098958 entries, 0 to 3098957
Data columns (total 2 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   bn_Keywords  object
 1   bn_Sentence  object
dtypes: object(2)
memory usage: 47.3+ MB
